# storeguard 통합 모델 학습 — Google Colab (A100)

무인매장 영상에서 **이상행동 3종 + 구매행동 6종**을 한 모델로 인식한다.

| 그룹 | 클래스 | 알림 |
|---|---|---|
| 음성 | `background_unlabeled` (라벨 구간 밖) | - |
| 이상행동 (238-2) | `fall` 전도, `broken` 파손, `theft` 절도 의심 | **알림 발생** |
| 구매행동 (238-1) | `moving` 매장이동, `select` 선택, `test` 시험, `buying` 구매, `return` 반품, `compare` 비교 | 알림 없음 (상태 표시만) |

## 왜 통합 모델인가 — 측정된 근거

이상행동만 학습한 기존 모델(`baseline_r2p1d`)을 **정상 구매행동 영상 133개(3.42시간)** 에
돌려 보니 **시간당 오경보 52.9건**이 나왔다(파손 40.9, 절도 12.0, 전도 0.0).
133개 중 104개 영상에서 오경보가 났다.

그 모델은 연출된 이상행동 영상만 보고 배웠고 거기서 "배경"은 행동 직전 구간뿐이었다.
**상품을 집고 계산하는 정상 행동을 한 번도 본 적이 없어서** 손을 뻗는 동작을
파손·절도로 읽는다. 구매행동을 별도 클래스로 넣어 이 문제를 줄이는 것이 목표다.

## 학습 데이터에 없는 클래스는 자동으로 빠진다

`train.py` 는 학습 클립에 한 번도 나오지 않은 클래스를 **출력에서 제외**한다.
학습되지 않은 출력 유닛이 탐지하는 척하지 않도록 하기 위해서다.
대시보드도 "이 모델이 다루지 않는 클래스"로 정직하게 표시한다.

## 준비 (로컬 PC)

전처리는 **로컬 CPU 가 가장 빠르다.** GPU 검토 결과와 이유는
`docs/preprocessing_optimization.md` 에 측정값과 함께 정리되어 있다. 요약하면
GTX 1050 Ti 의 NVDEC 는 엔진이 하나뿐이라 CPU 6워커보다 2.7배 느리고,
Colab GPU 를 쓰려면 원본 zip 670 GB 를 먼저 올려야 해서 업로드가 더 오래 걸린다.

```powershell
cd D:\computervision
$PY = ".\.venv\Scripts\python.exe"

# 1) 이상행동(238-2) 전체 + 구매행동(238-1) 라벨
& $PY -m storeguard.data.extract --dataset 238-2 --what all --split all
& $PY -m storeguard.data.extract --dataset 238-1 --what labels --split all

# 2) zip → 프레임 캐시 (한 단계. mp4 를 디스크에 쌓지 않는다)
#    끊겨도 다시 실행하면 이어서 한다.
& $PY -m storeguard.data.ingest --dry-run --cap moving=250 --max-per-class 0
& $PY -m storeguard.data.ingest --cap moving=250 --max-per-class 0 --workers 6

# 3) 인덱스 · 분할
& $PY -m storeguard.data.build_index --probe-video
& $PY -m storeguard.data.split

# 4) 업로드 묶음
& $PY -m storeguard.tools.make_colab_bundle
```

**현재 묶음: 코드 0.8 MB + 캐시 `cache_jpz.tar` 18.24 GB (영상 7,178개).**
예전에 올린 5.04 / 7.84 / 9.56 / 11.96 GB 짜리는 전부 구버전이니 **지우고 교체**할 것.

생성된 두 파일을 구글 드라이브 `MyDrive/storeguard/`(또는 공유 드라이브)에 올린다.
런타임 유형을 **A100** 으로 설정: `런타임 > 런타임 유형 변경 > GPU > A100`

## 학습에 들어가는 클립 (기대값)

| 클래스 | train | val | test | 손실 가중치 |
|---|---|---|---|---|
| 라벨구간외 | 23,095 | 4,908 | 5,717 | 1.000 |
| 전도 | 5,284 | 987 | 810 | 0.978 |
| 파손 | 5,508 | 900 | 800 | 0.938 |
| 절도 의심 | 5,341 | 927 | 795 | 0.967 |
| 매장이동 | 2,200 | 470 | 720 | 2.348 |
| 선택 | 4,652 | 1,009 | 1,304 | 1.110 |
| 시험 | 5,938 | 1,340 | 1,720 | 0.870 |
| 구매 | 5,820 | 1,420 | 1,800 | 0.888 |
| 반품 | 5,930 | 1,350 | 1,770 | 0.871 |
| 비교 | 5,820 | 1,360 | 1,830 | 0.888 |
| **합계** | **69,588** | 14,671 | 17,266 | |

**9개 클래스 중 8개의 가중치가 0.87~1.11 이다.** 인위적 보정이 거의 필요 없다는 뜻이다.
매장이동만 2.35 인데, 매장이 DYA/DYB 두 곳뿐이라 일부러 250개로 제한했기 때문이다.

영상 수는 train 4,911 / val 1,036 / test 1,226.
4번 셀 출력이 이 표와 크게 다르면 구버전 묶음을 올린 것이다.

## 0. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
if not torch.cuda.is_available():
    raise SystemExit('GPU 런타임이 아니다. 런타임 > 런타임 유형 변경 > GPU')
p = torch.cuda.get_device_properties(0)
cc = f'{p.major}.{p.minor}'
print(f'torch {torch.__version__} | {p.name} | VRAM {p.total_memory/1024**3:.1f} GB | compute {cc}')
if 'A100' not in p.name:
    print('\n[주의] A100 이 아니다. 배정은 보장되지 않는다.')
    print('      configs/colab_a100.yaml 대신 configs/colab.yaml 을 쓰고')
    print('      아래 1번 셀에서 batch_size 를 반드시 다시 재서 정할 것.')
CFG = 'configs/colab_a100.yaml' if 'A100' in p.name else 'configs/colab.yaml'
print('사용할 설정:', CFG)

## 1. 코드 가져오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/storeguard'
assert os.path.isdir(DRIVE), f'{DRIVE} 가 없다. 드라이브에 storeguard 폴더를 만들고 파일을 올릴 것.'
print(sorted(os.listdir(DRIVE)))

In [ ]:
%%bash
set -e
mkdir -p /content/storeguard
cd /content/storeguard
unzip -oq /content/drive/MyDrive/storeguard/storeguard_code.zip
mkdir -p reports
pip install -q pyyaml python-dotenv 2>&1 | tail -1 || true
ls

## 2. 배치 크기 실측

추측하지 않는다. 로컬 GTX 1050 Ti 는 **4.4 clips/s** 였다. 여기서 몇 배가 나오는지 본다.
`clips/s` 가 가장 높은 배치를 아래 3번 셀의 `BATCH` 에 적는다.

In [ ]:
%cd /content/storeguard
!python -m storeguard.tools.vram_probe --config {CFG} --batches 16,32,64,96,128

## 3. 데이터 가져오기

In [ ]:
%%bash
set -e
echo '--- Drive -> 로컬 디스크 복사 (Drive 를 직접 읽으면 훨씬 느리다)'
time cp /content/drive/MyDrive/storeguard/cache_jpz.tar /content/cache_jpz.tar
echo '--- 압축 해제'
time tar -xf /content/cache_jpz.tar -C /content/storeguard
ls /content/storeguard/data/cache | wc -l
du -sh /content/storeguard/data/cache
df -h /content | tail -1

## 4. 데이터 커버리지 확인 (학습 전에 반드시)

어떤 클래스가 실제로 들어왔는지 본다. 구매행동 데이터를 올리지 않았다면 여기서 드러난다.

In [ ]:
%cd /content/storeguard
import collections, json
from storeguard.config import Config
from storeguard.data.clips import build_clip_list
from storeguard.data.dataset import load_index, load_splits

cfg = Config.load(CFG)
rows, sp = load_index(cfg), load_splits(cfg)
ko = cfg.get('classes.ko', {})
print(f"{'분할':6s} {'영상':>6s} {'클립':>7s}   클래스별 클립")
present = set()
for k in ('train', 'val', 'test'):
    clips = build_clip_list(rows, sp.get(k, []), cfg)
    c = collections.Counter(s.label for s in clips)
    present |= set(c)
    print(f"{k:6s} {len(sp.get(k, [])):6d} {len(clips):7d}   "
          + ', '.join(f'{ko.get(n,n)} {c[n]}' for n in cfg.class_names if c[n]))

missing = [c for c in cfg.class_names if c not in present]
print()
if missing:
    print('[주의] 학습 데이터에 없는 클래스:', ', '.join(ko.get(m, m) for m in missing))
    print('       이 클래스들은 모델 출력에서 자동으로 제외된다.')
    print('       구매행동을 학습하려면 238-1 zip 을 모두 내려받아 다시 묶음을 만들 것.')
else:
    print('[확인] 설정의 모든 클래스가 학습 데이터에 존재한다.')

## 5. 학습

체크포인트는 드라이브에 저장한다. 세션이 끊겨도 `--resume` 으로 이어진다.

In [ ]:
BATCH   = 64        # 2번 셀에서 clips/s 가 가장 높았던 배치로 고칠 것
LR      = 0.0008    # batch 64 기준. 배치를 2배 키우면 lr 도 대략 2배
EPOCHS  = 12
WORKERS = 8
NAME    = 'unified_r2p1d'
RUNS    = '/content/drive/MyDrive/storeguard/runs'

import shlex
cmd = (f'python -m storeguard.train --config {CFG} --name {NAME} '
       f'--set train.batch_size={BATCH} --set train.lr={LR} '
       f'--set train.epochs={EPOCHS} --set train.num_workers={WORKERS} '
       f'--set paths.runs={shlex.quote(RUNS)}')
print(cmd)

In [ ]:
%cd /content/storeguard
!{cmd}

### 세션이 끊겼다면
0~4번 셀을 다시 돌린 뒤 아래를 실행한다.

In [ ]:
%cd /content/storeguard
!{cmd} --resume

## 6. 평가

시험 분할은 배포 Validation 세트다. 모델 선택이나 임계값 조정에 쓰지 않는다.

In [ ]:
%cd /content/storeguard
!python -m storeguard.evaluate --run {NAME} --runs-dir {RUNS} --split test
!python -m storeguard.tools.make_report --run {NAME} --runs-dir {RUNS} --reports /content/storeguard/reports

In [ ]:
import json, pathlib
p = pathlib.Path(RUNS) / NAME / 'eval_test.json'
r = json.loads(p.read_text(encoding='utf-8'))
c = r['클립단위']
print(f"클립 단위  정확도 {c['accuracy']}  macro-F1 {c['macro_f1']}")
for k, v in c['per_class'].items():
    print(f"  {k:22s} P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f} n={v['support']}")
e = r.get('사건단위')
if isinstance(e, dict):
    print('\n사건 단위 (알림 대상 클래스만)')
    for k, v in e['클래스별'].items():
        print(f"  {k:8s} 탐지율 {v['사건단위_탐지율']}  오탐 {v['오탐(FP)']}  "
              f"지연중앙값 {v['지연_초']['median']}초")

## 7. 결과 회수

`runs` 는 이미 드라이브에 있다. 로컬 PC 에서 `MyDrive/storeguard/runs/unified_r2p1d` 를
`D:\computervision\runs\` 아래로 내려받은 뒤 대시보드를 띄우면 된다.

```powershell
cd D:\computervision
.\scripts\run_dashboard.cmd --loop
```

`run_dashboard.cmd` 는 `runs\baseline_r2p1d` 를 찾는다. 새 모델을 쓰려면 직접 지정한다.

```powershell
.\.venv\Scripts\python.exe -m storeguard.server.app --run unified_r2p1d --loop
```

In [ ]:
!cp -r /content/storeguard/reports /content/drive/MyDrive/storeguard/reports_colab
!ls -la {RUNS}/{NAME}
!cat {RUNS}/{NAME}/classes.json

## 주의할 점

- **프레임 캐시는 손실 압축(JPEG 품질 90)이다.** 원본 `.npy` 와 비교해 실측한 결과,
  예측 클래스 일치율은 품질 95 와 **똑같이 97.70 %** 이고 점수 차이만 0.0284 → 0.0327 로
  아주 조금 커진다. 용량은 13.2 GB → **9.56 GB** 로 줄어 구글 드라이브 무료 용량에 여유가 생긴다.
  학습과 평가를 같은 형식으로 하면 문제없다.
- **원천 영상의 FPS 가 클래스마다 다르다.** 이상행동·매장이동은 3 fps, 구매행동 5종은 10 fps 다.
  전처리가 전부 `data.sample_fps`(기본 3) 로 시간 리샘플링하고,
  **라벨 프레임 번호도 같은 배율로 변환한다**(`clips.label_scale`).
  이 변환이 없으면 구매행동 라벨이 3배 뒤로 밀려 통째로 어긋난다.
  구매행동의 시간 해상도는 1/3 로 줄어든다. 손 동작이 세밀한 선택/구매/반품 구분에 불리할 수 있다.
- **선택(select)은 어렵다.** 이벤트가 1.4~3.0초인데 클립은 5.33초라 클립 대부분이 선택 아닌 장면이다.
  학습 클립도 1,430개로 행동 클래스 중 가장 적다(가중치 2.09).
- **매장이동(moving)은 5분 영상 전체가 한 라벨이다.** 사람이 없는 구간도 많아
  배경 클래스와 혼동될 수 있다. 평가에서 이 둘의 혼동행렬을 꼭 확인할 것.
  매장도 DYA/DYB **두 곳뿐**이라 새 매장 일반화에 특히 약하다.
- **배치를 키우면 학습률도 같이 올린다.** 배치만 키우면 학습이 덜 진행된다.
- **test 분할은 한 번만 쓴다.** 임계값 조정과 모델 선택은 val 로만 한다.
- **A100 이 배정되지 않을 수 있다.** 유료 요금제도 GPU 종류를 보장하지 않는다.

## 학습이 끝나면 꼭 할 것 — 오경보 개선폭 측정

통합 모델이 정말 오경보를 줄였는지 **같은 도구로 다시 재서** 비교한다.
기존 이상행동 전용 모델은 정상 영상에서 시간당 **52.9건**이었다.

```powershell
# 로컬 PC 에서 (드라이브의 runs/unified_r2p1d 를 D:\computervision\runs\ 로 내려받은 뒤)
cd D:\computervision
.\.venv\Scripts\python.exe -m storeguard.tools.false_alarm --run unified_r2p1d --compare baseline_r2p1d
```

정확도나 F1 만 보고 판단하지 말 것. **이 수치가 크게 떨어져야** 통합 모델을 만든 목적이 달성된 것이다.